<a href="https://colab.research.google.com/github/JoviWZhu/20206RAG/blob/RAG-Quiz/FineWeb_Edu_RAG_Quiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets sentence-transformers faiss-cpu huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.1 MB/s eta 0:00:00


In [3]:
import os
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from huggingface_hub import InferenceClient
import random

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
# Replace with your free Hugging Face Read Token

from google.colab import userdata
HF_TOKEN_DEV = userdata.get('HF_TOKEN_DEV')

client = InferenceClient(token=HF_TOKEN_DEV)


print("⚡ Step 1: Loading an educational pool from FineWeb-Edu...")
# We load a slightly larger sample (500 documents) so the quiz has diverse topics
dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
#dataset_head = dataset.take(500)

# Pick a random starting point somewhere in the massive dataset stream
random_skip_offset = random.randint(0, 10000)

# Stream 100 rows starting from that random location
dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
dataset_head = dataset.skip(random_skip_offset).take(500)

documents = [doc["text"] for doc in dataset_head]
print(f"✅ Loaded {len(documents)} educational passages into the quiz pool.")

# Global variables to hold the current quiz state
current_context = ""
current_question = ""



⚡ Step 1: Loading an educational pool from FineWeb-Edu...


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

✅ Loaded 500 educational passages into the quiz pool.


In [ ]:
# Take the first 3 examples from the stream
for i, example in enumerate(dataset.take(3)):
    print(f"--- Example {i+1} ---")
    print(f"Text snippet: {example['text'][:100]}...") # Prints first 200 characters
    print(f"Educational score: {example['score']}")
    print(f"Source URL: {example['url']}\n")
    print(f"Language: {example['language']}\n")
    print(f"Language Score: {example['language_score']}\n")

--- Example 1 ---
Text snippet: The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are rea...
Educational score: 2.75
Source URL: http://austenauthors.net/the-independent-jane

Language: en

Language Score: 0.9743200540542603

--- Example 2 ---
Text snippet: Taking Play Seriously
By ROBIN MARANTZ HENIG
Published: February 17, 2008
On a drizzly Tuesday night...
Educational score: 2.5625
Source URL: http://query.nytimes.com/gst/fullpage.html?res=9404E7DA1339F934A25751C0A96E9C8B63&scp=2&sq=taking%20play%20seriously&st=cse

Language: en

Language Score: 0.9614589214324951

--- Example 3 ---
Text snippet: How do you get HIV?
HIV can be passed on when infected bodily fluid, such as blood or semen, is pass...
Educational score: 3.125
Source URL: http://www.childline.org.uk/Explore/SexRelationships/Pages/HIVAIDS.aspx

Language: en

Language Score: 0.9667569994926453



In [ ]:
high_quality_dataset = dataset.filter(lambda x: x["score"] > 5)

In [ ]:
for i, example in enumerate(high_quality_dataset.take(3)):
    print(f"--- Example {i+1} ---")
    print(f"Text snippet: {example['text'][:100]}...") # Prints first 200 characters
    print(f"Educational score: {example['score']}")
    print(f"Source URL: {example['url']}\n")
    print(f"Language: {example['language']}\n")
    print(f"Language Score: {example['language_score']}\n")

--- Example 1 ---
Text snippet: This topic introduces the fundamentals of radical expressions. In this six lesson series, the studen...
Educational score: 5.0625
Source URL: http://ilearn.com/main/ilearntopics/algebra1/radicals-i.html

Language: en

Language Score: 0.9292330145835876

--- Example 2 ---
Text snippet: Understanding Systems of Inequalities
In a system of inequalities, you see more than one inequality ...
Educational score: 5.03125
Source URL: http://www.dummies.com/how-to/content/understanding-systems-of-inequalities.navId-611287.html

Language: en

Language Score: 0.9463697671890259

--- Example 3 ---
Text snippet: The learning goals for this section are for students to understand the need for (1) brackets in expr...
Educational score: 5.34375
Source URL: https://www.jeremybarr.ca/blog/2014/09/11/order-of-operations

Language: en

Language Score: 0.9356819987297058



In [ ]:
shuffled_dataset = dataset.shuffle(buffer_size=10_000, seed=42)

In [ ]:
for i, example in enumerate(shuffled_dataset.take(3)):
    print(f"--- Example {i+1} ---")
    print(f"Text snippet: {example['text'][:100]}...") # Prints first 200 characters
    print(f"Educational score: {example['score']}")
    print(f"Source URL: {example['url']}\n")
    print(f"Language: {example['language']}\n")
    print(f"Language Score: {example['language_score']}\n")
    print(f"Tokenize Size: {example['token_count']}\n")


--- Example 1 ---
Text snippet: The entrance to Simbol Materials' Calipatria demonstration facility is seen on Wednesday, January 8,...
Educational score: 3.484375
Source URL: http://archive.desertsun.com/article/20140222/BUSINESS0302/302220055/Simbol-Materials-lithium-extraction-Salton-Sea

Language: en

Language Score: 0.9487097859382629

Tokenize Size: 3553

--- Example 2 ---
Text snippet: Surface sediments of mangrove, freshwater wetland and rainforest sites in northeast Queensland were ...
Educational score: 3.1875
Source URL: https://research.monash.edu/en/publications/modern-pollen-deposition-in-the-tropical-lowlands-of-northeast-qu

Language: en

Language Score: 0.9484227895736694

Tokenize Size: 172

--- Example 3 ---
Text snippet: Successful first flight trial completion of jet-powered UAV
13 December 2017
BAE Systems and Manches...
Educational score: 2.953125
Source URL: http://www.dpaonthenet.net/article/143315/Successful-first-flight-trial-completion-of-jet-powered-UAV.asp

In [8]:
# ==========================================
# 2. FUNCTION TO GENERATE A QUIZ QUESTION
# ==========================================
def generate_quiz_question():
    global current_context, current_question

    print("\n⚡ Selecting random educational context and creating a question...")

    # Pick a random passage from our FineWeb-Edu pool
    current_context = random.choice(documents)

    system_prompt = (
        "You are an academic professor. Your job is to read the provided text context and create ONE challenging, "
        "open-ended question based strictly on the facts inside the text. Do not provide options or the answer. "
        "Just output the question directly."
    )

    user_prompt = f"Context from FineWeb-Edu:\n{current_context}\n\nGenerate ONE question based on this context:"

    try:
        response = client.chat_completion(
            model="meta-llama/Llama-3.1-8B-Instruct",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=150,
            temperature=0.7 # Slightly higher temperature for interesting questions
        )

        # Extract the question text cleanly
        if hasattr(response, 'choices') and len(response.choices) > 0:
            current_question = response.choices[0].message.content
        else:
            current_question = str(response)

        print(f"\n📝 [PROFESSOR'S READ MATERIAL]:\n{current_context}\n")
        print(f"\n📝 [PROFESSOR'S QUESTION]:\n{current_question}\n")
        print("👉 Run the next block or function `grade_quiz_answer('your answer here')` to submit your response!")

    except Exception as e:
        print(f"❌ Error generating question: {e}")

# ==========================================
# 3. FUNCTION TO GRADE THE USER'S RESPONSE
# ==========================================
def grade_quiz_answer(user_answer):
    global current_context, current_question

    if not current_context or not current_question:
        print("❌ No active quiz question found! Run generate_quiz_question() first.")
        return

    print(f"\n⚡ Grading your answer: '{user_answer}'...")

    system_prompt = (
        "You are an academic grader. Evaluate the user's answer against the original textbook source context "
        "and the question asked. State clearly at the beginning if they are [CORRECT], [PARTIALLY CORRECT], or [INCORRECT]. "
        "Then, explain why based strictly on the provided context, and point out any missing or inaccurate information. "
        "Be encouraging but rigorous."
    )

    user_prompt = (
        f"Original Source Context:\n{current_context}\n\n"
        f"Question Asked: {current_question}\n\n"
        f"User's Answer: {user_answer}\n\n"
        f"Provide the grade and explanation:"
    )

    try:
        response = client.chat_completion(
            model="meta-llama/Llama-3.1-8B-Instruct",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=400,
            temperature=0.2 # Low temperature for accurate factual grading
        )

        print("\n🎓 [GRADER FEEDBACK]:")
        if hasattr(response, 'choices') and len(response.choices) > 0:
            print(response.choices[0].message.content)
        else:
            print(response)

    except Exception as e:
        print(f"❌ Error communicating with Grading API: {e}")

In [9]:
# 1. Generate the question
generate_quiz_question()

# 2. Open an interactive text box right inside Colab for your answer
your_response = input("Type your answer to the question above: ")

# 3. Grade your answer instantly using the source data
grade_quiz_answer(your_response)


⚡ Selecting random educational context and creating a question...

📝 [PROFESSOR'S READ MATERIAL]:
Mon 20 Jul 2009
As far as scary-looking fish go, the adult sea lamprey is right up there with the sharks. It’s about 3 feet long, with concentric rings of teeth inside a jawless, suction-cup mouth. After trapping its prey by sucking, the lamprey scrapes off the skin using a tooth at the end of its tongue, and then sucks out the blood. Since the lamprey has good taste in fish – preferring trout, salmon, bass –it’s the target of a federal lampricide program in the Great Lakes region, where it has nearly devoured some of these stocks.
But no creature is all bad. “The lamprey has a really beautiful spinal cord,” says Ona Bloom, an MBL Research Fellow from The Feinstein Institute for Medical Research. Together with Jennifer Morgan (Univ. of Texas at Austin) and David Parker (Univ. of Cambridge), who are recipients of an MBL sponsored Albert and Ellen Grass Faculty Research Grant, Bloom and her

In [10]:
# ==========================================
# 2. FUNCTION TO GENERATE A QUIZ QUESTION
# ==========================================
def generate_quiz_question():
    global current_context, current_question

    print("\n⚡ Selecting random educational context and creating a question...")

    # 1. Pick a random passage from your 100 FineWeb-Edu documents
    current_context = random.choice(documents)

    # 2. Define the different types of questions you want to ask
    quiz_angles = [
        "Create a challenging multiple-choice question focusing on an exact definition.",
        "Create a true/false question targeting a common misconception in this text.",
        "Create an 'Explain the Cause' question based on the events or mechanisms explained.",
        "Create a question that applies the concept in this text to a real-world scenario."
    ]

    # 3. Pick a random question angle
    selected_angle = random.choice(quiz_angles)
    print(f"🎯 Selected Angle: {selected_angle}") # Diagnostic message

    # 4. Inject the chosen angle directly into the system prompt instructions
    system_prompt = (
        f"You are an academic professor. Read the provided text context and fulfill this requirement: {selected_angle} "
        "Do not provide the answer key or a grading matrix. Output ONLY the question itself clearly."
    )

    user_prompt = f"Context from FineWeb-Edu:\n{current_context}\n\nGenerate the question now:"

    try:
        response = client.chat_completion(
            model="meta-llama/Llama-3.1-8B-Instruct",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=150,
            temperature=0.7 # Slightly higher temperature for interesting questions
        )

        # Extract the question text cleanly
        if hasattr(response, 'choices') and len(response.choices) > 0:
            current_question = response.choices[0].message.content
        else:
            current_question = str(response)

        print(f"\n📝 [PROFESSOR'S READ MATERIAL]:\n{current_context}\n")
        print(f"\n📝 [PROFESSOR'S QUESTION]:\n{current_question}\n")
        print("👉 Run the next block or function `grade_quiz_answer('your answer here')` to submit your response!")

    except Exception as e:
        print(f"❌ Error generating question: {e}")

# ==========================================
# 3. FUNCTION TO GRADE THE USER'S RESPONSE
# ==========================================
def grade_quiz_answer(user_answer):
    global current_context, current_question

    if not current_context or not current_question:
        print("❌ No active quiz question found! Run generate_quiz_question() first.")
        return

    print(f"\n⚡ Grading your answer: '{user_answer}'...")

    system_prompt = (
        "You are an academic grader. Evaluate the user's answer against the original textbook source context "
        "and the question asked. State clearly at the beginning if they are [CORRECT], [PARTIALLY CORRECT], or [INCORRECT]. "
        "Then, explain why based strictly on the provided context, and point out any missing or inaccurate information. "
        "Be encouraging but rigorous."
    )

    user_prompt = (
        f"Original Source Context:\n{current_context}\n\n"
        f"Question Asked: {current_question}\n\n"
        f"User's Answer: {user_answer}\n\n"
        f"Provide the grade and explanation:"
    )

    try:
        response = client.chat_completion(
            model="meta-llama/Llama-3.1-8B-Instruct",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=400,
            temperature=0.2 # Low temperature for accurate factual grading
        )

        print("\n🎓 [GRADER FEEDBACK]:")
        if hasattr(response, 'choices') and len(response.choices) > 0:
            print(response.choices[0].message.content)
        else:
            print(response)

    except Exception as e:
        print(f"❌ Error communicating with Grading API: {e}")

In [11]:
# 1. Generate the question
generate_quiz_question()

# 2. Open an interactive text box right inside Colab for your answer
your_response = input("Type your answer to the question above: ")

# 3. Grade your answer instantly using the source data
grade_quiz_answer(your_response)


⚡ Selecting random educational context and creating a question...
🎯 Selected Angle: Create a true/false question targeting a common misconception in this text.

📝 [PROFESSOR'S READ MATERIAL]:
One of the great bits of repartee in The King’s Speech comes as the maverick Australian speech therapist, Lionel Logue, is just getting to know His Royal Highness Prince Albert, the stammering Duke of York:
Logue: “Surely a prince’s brain knows what his mouth’s doing?” Bertie: “You’re obviously not well acquainted with many royal princes.”
No one could have imagined any such dialogue involving Archduke Otto von Habsburg, who died on July 4—not because the archduke was a fearsome personality, but because he was a pre-eminently intelligent and decent man.
The full name he was given at his baptism in 1912—Franz Josef Otto Robert Maria Anton Karl Max Heinrich Sixtus Xavier Felix Renatus Ludwig Gaetan Pius Ignatius—speaks volumes about the history of his family, whose rule over central Europe extended

In [12]:
# 1. Generate the question
generate_quiz_question()

# 2. Open an interactive text box right inside Colab for your answer
your_response = input("Type your answer to the question above: ")

# 3. Grade your answer instantly using the source data
grade_quiz_answer(your_response)


⚡ Selecting random educational context and creating a question...
🎯 Selected Angle: Create a true/false question targeting a common misconception in this text.

📝 [PROFESSOR'S READ MATERIAL]:
I was helping a friend with setting up her iMac computer yesterday for genealogy and she said something that caught my attention. She said, "I have to learn what all the genealogy terms mean." That brought home a common problem with all specializations; learning the jargon.
Jargon is a really a technical term I became acquainted with during my graduate studies in Linguistics at the University of Utah. It is defined as special words or expressions that are used by a particular profession or group and are difficult for others to understand. The main problem with jargon is that we often do not even know we are using it.
Take for example, the word "genealogy" itself, as in "I am doing my genealogy" or "I am working on my genealogy." What does that statement communicate to a non-genealogist? And somet